# 🎨 Module 1.3: Color Spaces — The Language of Color

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_01_Image_Foundations/03_Color_Spaces/color_spaces.ipynb)

---

## 🎯 Learning Objectives
1. Understand RGB, HSV, LAB, YCbCr color spaces mathematically
2. Convert between color spaces with full derivation
3. Visualize why different color spaces matter for RL
4. Choose the right color space for different RL tasks

---

In [ ]:
!pip install numpy matplotlib opencv-python-headless scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import cv2

print("✅ Ready!")

In [ ]:
# Download REAL open-source dataset
from skimage import data
import torchvision

# Real images from scikit-image (built-in, no download needed)
astronaut_img = data.astronaut()       # 512x512x3 real photo
camera_img = data.camera()             # 512x512 grayscale real photo  
coins_img = data.coins()               # Real coins photograph
coffee_img = data.coffee()             # 400x600x3 real photo

# MNIST - 70,000 real handwritten digits (auto-downloads ~11MB)
mnist_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True)
print(f"✅ MNIST loaded: {len(mnist_dataset)} real handwritten digit images (28x28)")
print(f"✅ scikit-image loaded: astronaut {astronaut_img.shape}, camera {camera_img.shape}")

## Deep Mathematical Derivation: Color Space Transformations

### Step 1: RGB to Grayscale — Perceptual Luminance
The conversion is NOT a simple average. It uses perceptual weights from CIE 1931:
$$Y = 0.2126 R + 0.7152 G + 0.0722 B$$

**Proof of why these weights are NOT equal:**
Human cone cell sensitivity peaks: L-cones (red, ~564nm), M-cones (green, ~534nm), S-cones (blue, ~420nm). Green dominates because M-cones have the broadest response curve and highest density on the retina.

**Derivation from CIE color matching functions:**
$$Y = \int_0^\infty S(\lambda) \cdot \bar{y}(\lambda) \, d\lambda$$
where $\bar{y}(\lambda)$ is the CIE luminous efficiency function. Discretizing over R, G, B primaries yields the ITU-R BT.709 weights above. $\blacksquare$

### Step 2: RGB → HSV Conversion (Complete Derivation)

**Given:** $(R, G, B) \in [0, 1]^3$. Let $C_{\max} = \max(R, G, B)$, $C_{\min} = \min(R, G, B)$, $\Delta = C_{\max} - C_{\min}$.

**Value:** $V = C_{\max}$ (trivial — brightest channel)

**Saturation:**
$$S = \begin{cases} 0 & \text{if } C_{\max} = 0 \\ \frac{\Delta}{C_{\max}} & \text{otherwise} \end{cases}$$

**Proof:** Saturation measures "how far from gray." A gray pixel has $R = G = B$, so $\Delta = 0$. Maximum saturation ($S=1$) occurs when $C_{\min} = 0$ (one channel is zero). $\blacksquare$

**Hue (the complex one):**
$$H = \begin{cases} 0° & \text{if } \Delta = 0 \\ 60° \times \frac{G - B}{\Delta} \mod 6 & \text{if } C_{\max} = R \\ 60° \times \left(\frac{B - R}{\Delta} + 2\right) & \text{if } C_{\max} = G \\ 60° \times \left(\frac{R - G}{\Delta} + 4\right) & \text{if } C_{\max} = B \end{cases}$$

**Derivation:** The hexagonal color wheel has 6 sectors of 60° each. The dominant channel determines the sector; the ratio of the other two channels determines position within the sector. $\blacksquare$

### Step 3: RGB → CIE LAB (Perceptually Uniform)

**Step 3a:** RGB → XYZ (linear transformation):
$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = \begin{bmatrix} 0.4124 & 0.3576 & 0.1805 \\ 0.2126 & 0.7152 & 0.0722 \\ 0.0193 & 0.1192 & 0.9505 \end{bmatrix} \begin{bmatrix} R_{\text{linear}} \\ G_{\text{linear}} \\ B_{\text{linear}} \end{bmatrix}$$

**Step 3b:** XYZ → LAB (non-linear, cube-root compression):
$$L^* = 116 \cdot f\left(\frac{Y}{Y_n}\right) - 16, \quad a^* = 500\left[f\left(\frac{X}{X_n}\right) - f\left(\frac{Y}{Y_n}\right)\right], \quad b^* = 200\left[f\left(\frac{Y}{Y_n}\right) - f\left(\frac{Z}{Z_n}\right)\right]$$

where the nonlinear function is:
$$f(t) = \begin{cases} t^{1/3} & t > \delta^3 \\ \frac{t}{3\delta^2} + \frac{4}{29} & t \leq \delta^3 \end{cases}, \quad \delta = \frac{6}{29}$$

**Proof of perceptual uniformity:** The cube root mimics the human visual system's logarithmic response (Stevens' power law with exponent ≈ 1/3). A distance $\Delta E = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$ of ~2.3 is the just-noticeable difference (JND). $\blacksquare$

### Step 4: Why Color Spaces Matter for RL
**Theorem:** The choice of color space affects RL convergence rate.

In RGB: Euclidean distance $\|\mathbf{p}_1 - \mathbf{p}_2\|_2$ does NOT correspond to perceptual difference.

In LAB: $\Delta E_{ab}$ IS perceptually uniform → better reward signals for image quality.

**RL implication:** Using LAB for reward computation gives the agent a reward signal aligned with human perception, leading to faster convergence and better subjective results.

## CIE XYZ Color Space — Derivation from Color Matching Functions

The CIE XYZ system is the mathematical foundation of ALL color spaces. Every other space (RGB, HSV, LAB) is derived from it.

### Step 1: The Physics of Color Perception

Human color vision begins with three types of cone cells, each with a different spectral sensitivity $\bar{l}(\lambda), \bar{m}(\lambda), \bar{s}(\lambda)$ (long, medium, short wavelength cones).

The response of each cone type to a spectral power distribution $S(\lambda)$:
$$L = \int_0^\infty S(\lambda) \cdot \bar{l}(\lambda) \, d\lambda, \quad M = \int_0^\infty S(\lambda) \cdot \bar{m}(\lambda) \, d\lambda, \quad S = \int_0^\infty S(\lambda) \cdot \bar{s}(\lambda) \, d\lambda$$

### Step 2: CIE 1931 Color Matching Functions

The CIE defined standard observers through color matching experiments. The tristimulus values are:

$$X = \int_0^\infty S(\lambda) \cdot \bar{x}(\lambda) \, d\lambda, \quad Y = \int_0^\infty S(\lambda) \cdot \bar{y}(\lambda) \, d\lambda, \quad Z = \int_0^\infty S(\lambda) \cdot \bar{z}(\lambda) \, d\lambda$$

where $\bar{x}(\lambda), \bar{y}(\lambda), \bar{z}(\lambda)$ are the CIE color matching functions, chosen so that:
- $\bar{y}(\lambda)$ equals the luminous efficiency function (photopic vision)
- All tristimulus values are non-negative for all visible spectra
- Equal-energy white maps to $X = Y = Z$

### Step 3: RGB to XYZ Transformation Matrix Derivation

For sRGB with D65 illuminant, the transformation is derived from the chromaticity coordinates of the RGB primaries:

| Primary | $x$ | $y$ |
|:--------|:----|:----|
| Red | 0.6400 | 0.3300 |
| Green | 0.3000 | 0.6000 |
| Blue | 0.1500 | 0.0600 |
| D65 White | 0.3127 | 0.3290 |

From chromaticity $(x, y)$ to XYZ: $X = x/y$, $Y = 1$, $Z = (1-x-y)/y$

The column vectors of the RGB→XYZ matrix are the XYZ coordinates of each primary, scaled so that RGB=(1,1,1) maps to the white point:

$$\mathbf{M}_{RGB \to XYZ} = \begin{bmatrix} 0.4124564 & 0.3575761 & 0.1804375 \\ 0.2126729 & 0.7151522 & 0.0721750 \\ 0.0193339 & 0.1191920 & 0.9503041 \end{bmatrix}$$

### Step 4: Proof That Y Channel Equals Luminance

By construction, the second row of $\mathbf{M}$ contains the luminance coefficients:
$$Y = 0.2126 R + 0.7152 G + 0.0722 B$$

This matches the photopic luminous efficiency function because $\bar{y}(\lambda)$ was specifically chosen to equal the CIE luminosity function $V(\lambda)$. This is why the grayscale conversion formula uses these specific weights, NOT equal weights of $1/3$. $\blacksquare$

### Step 5: Metamerism — A Mathematical Consequence

**Definition:** Two different spectra $S_1(\lambda)$ and $S_2(\lambda)$ are metamers if they produce identical tristimulus values.

**Proof of existence:** Since integration $\int S(\lambda) \bar{x}(\lambda) d\lambda$ is a projection from infinite-dimensional function space to $\mathbb{R}^3$, the kernel is infinite-dimensional. Therefore infinitely many spectra map to the same $(X, Y, Z)$. $\blacksquare$

This is why a computer monitor (emitting 3 wavelengths) can match the appearance of a sunset (continuous spectrum) — they are metamers.

## RGB to HSV — Geometric Derivation on the Hexcone

The HSV color space maps the RGB cube to a more intuitive cylindrical coordinate system. The conversion involves a beautiful piece of computational geometry.

### Step 1: The RGB Cube and the Neutral Axis

In the RGB unit cube $[0,1]^3$, the neutral (gray) axis runs from $(0,0,0)$ (black) to $(1,1,1)$ (white). Points on this axis satisfy $R = G = B$.

**Key observation:** "Color" (hue and saturation) is the deviation from this neutral axis.

### Step 2: Projecting onto the Hexagonal Cross-Section

Slice the RGB cube perpendicular to the neutral axis at height $V = \max(R,G,B)$. The cross-section at constant $V$ is a hexagon with vertices at the six pure hues: Red, Yellow, Green, Cyan, Blue, Magenta.

**Why hexagonal?** The cube $[0,V]^3$ has vertices at distance $V$ from the origin along each axis. The cross-section perpendicular to $(1,1,1)$ intersects 6 edges of the cube, forming a regular hexagon.

### Step 3: Hue Angle Derivation

**Projection to chromaticity plane:** Define coordinates in the plane perpendicular to $(1,1,1)$:

$$\alpha = R - \frac{1}{2}G - \frac{1}{2}B, \quad \beta = \frac{\sqrt{3}}{2}(G - B)$$

These are obtained by projecting $(R,G,B)$ onto two orthonormal vectors in the plane $\perp (1,1,1)$:

$$\mathbf{e}_1 = \frac{1}{\sqrt{6}}(2, -1, -1)^T, \quad \mathbf{e}_2 = \frac{1}{\sqrt{2}}(0, 1, -1)^T$$

The hue angle is:
$$H = \text{atan2}(\beta, \alpha) = \text{atan2}\left(\frac{\sqrt{3}}{2}(G-B), \; R - \frac{G+B}{2}\right)$$

**Proof of equivalence to the piecewise formula:**

When $R = C_{\max}$: $\alpha > 0$ and $\beta = \frac{\sqrt{3}}{2}(G-B)$. Then $H = \arctan\left(\frac{\sqrt{3}(G-B)}{2R - G - B}\right)$. Since $R = C_{\max}$ and $\Delta = C_{\max} - C_{\min}$, we can show $2R - G - B = 2\Delta - (G-B)... $ through algebraic manipulation, this reduces to $H = 60° \cdot \frac{G-B}{\Delta}$. $\blacksquare$

### Step 4: Saturation as Radial Distance

$$S = \frac{\sqrt{\alpha^2 + \beta^2}}{V} = \frac{\Delta}{C_{\max}} = 1 - \frac{C_{\min}}{C_{\max}}$$

**Proof:** The maximum possible radial distance at height $V$ is $V$ (when one channel is $V$ and another is 0). The actual radial distance is proportional to $\Delta = C_{\max} - C_{\min}$. Normalizing: $S = \Delta / V$. $\blacksquare$

### Step 5: The Hexcone Model

HSV maps to a hexagonal cone (hexcone):
- **Apex** (S=0, V=0): black point
- **Central axis** (S=0): gray values from black to white  
- **Base circle** (V=1, S=1): fully saturated colors
- **Top face** (V=1): the color hexagon

The geometry makes HSV intuitive for an RL agent:
- Rotating around the cone (changing H) selects hue without affecting brightness
- Moving radially (changing S) adjusts color purity independently
- Moving vertically (changing V) adjusts brightness independently

This independence of dimensions makes HSV a natural action space for color manipulation tasks — each action (adjust H, S, or V) has a predictable, isolated effect.

## RGB to LAB — The Complete Transform Chain with Numerical Examples

The RGB→LAB conversion is a multi-step process involving both linear and nonlinear transformations. Here we trace through the complete chain with explicit computations.

### Step 1: Linearize sRGB (Gamma Decode)

First, convert sRGB (gamma-encoded) to linear RGB:

$$R_{\text{lin}} = \begin{cases} R_{\text{sRGB}} / 12.92 & R_{\text{sRGB}} \leq 0.04045 \\ \left(\frac{R_{\text{sRGB}} + 0.055}{1.055}\right)^{2.4} & R_{\text{sRGB}} > 0.04045 \end{cases}$$

(Same for G and B channels.)

**Numerical example:** For a medium orange pixel $\text{sRGB} = (200, 120, 50)/255 = (0.784, 0.471, 0.196)$:
$$R_{\text{lin}} = \left(\frac{0.784 + 0.055}{1.055}\right)^{2.4} = 0.580$$
$$G_{\text{lin}} = \left(\frac{0.471 + 0.055}{1.055}\right)^{2.4} = 0.188$$
$$B_{\text{lin}} = \left(\frac{0.196 + 0.055}{1.055}\right)^{2.4} = 0.031$$

### Step 2: Linear RGB → XYZ

$$\begin{bmatrix} X \\ Y \\ Z \end{bmatrix} = \begin{bmatrix} 0.4124 & 0.3576 & 0.1805 \\ 0.2126 & 0.7152 & 0.0722 \\ 0.0193 & 0.1192 & 0.9505 \end{bmatrix} \begin{bmatrix} 0.580 \\ 0.188 \\ 0.031 \end{bmatrix} = \begin{bmatrix} 0.312 \\ 0.262 \\ 0.040 \end{bmatrix}$$

### Step 3: XYZ → LAB (with D65 Reference White)

Reference white D65: $(X_n, Y_n, Z_n) = (0.9505, 1.0000, 1.0890)$

Compute normalized ratios: $X/X_n = 0.328$, $Y/Y_n = 0.262$, $Z/Z_n = 0.037$

Apply the nonlinear function $f(t) = t^{1/3}$ for $t > 0.008856$:
$$f(X/X_n) = 0.328^{1/3} = 0.689$$
$$f(Y/Y_n) = 0.262^{1/3} = 0.640$$
$$f(Z/Z_n) = 0.037^{1/3} = 0.333$$

### Step 4: Final LAB Values

$$L^* = 116 \times 0.640 - 16 = 58.2$$
$$a^* = 500 \times (0.689 - 0.640) = 24.5$$
$$b^* = 200 \times (0.640 - 0.333) = 61.4$$

**Result:** The orange pixel $(200, 120, 50)_{\text{sRGB}}$ maps to $(58.2, 24.5, 61.4)_{\text{LAB}}$

**Interpretation:**
- $L^* = 58.2$: medium lightness (0 = black, 100 = white)
- $a^* = 24.5$: slightly reddish (positive $a^*$ = red)
- $b^* = 61.4$: strongly yellowish (positive $b^*$ = yellow)

This matches our perception: the pixel is a warm orange (red + yellow, medium brightness).

### Step 5: Why Each Step Is Necessary

1. **Gamma decode:** Required because arithmetic in gamma space is nonlinear — adding two gamma-encoded values doesn't correspond to adding two light intensities
2. **RGB→XYZ:** Maps device-specific primaries to device-independent reference
3. **XYZ→LAB:** Applies perceptual nonlinearity (cube root) to achieve approximate perceptual uniformity

Skipping any step produces incorrect color comparisons — a common bug in image processing pipelines that directly impacts RL reward accuracy.

## 1. RGB Color Space

### Mathematical Model:
$$\mathbf{c}_{RGB} = (R, G, B) \in [0, 255]^3$$

The RGB cube represents all possible colors in a 3D space:
- Origin $(0,0,0)$ = Black
- $(255,255,255)$ = White
- Axes represent primary colors

### Problem for RL:
RGB channels are **correlated** — changing one channel often requires adjusting others to maintain natural appearance. This makes the RL action space complex.

## 2. HSV Color Space

### RGB to HSV Conversion:

Let $R', G', B' = R/255, G/255, B/255$

$$C_{max} = \max(R', G', B'), \quad C_{min} = \min(R', G', B')$$
$$\Delta = C_{max} - C_{min}$$

**Hue:**
$$H = \begin{cases} 0° & \text{if } \Delta = 0 \\ 60° \times \frac{G'-B'}{\Delta} \mod 6 & \text{if } C_{max} = R' \\ 60° \times \left(\frac{B'-R'}{\Delta} + 2\right) & \text{if } C_{max} = G' \\ 60° \times \left(\frac{R'-G'}{\Delta} + 4\right) & \text{if } C_{max} = B' \end{cases}$$

**Saturation:**
$$S = \begin{cases} 0 & \text{if } C_{max} = 0 \\ \frac{\Delta}{C_{max}} & \text{otherwise} \end{cases}$$

**Value:**
$$V = C_{max}$$

### Why HSV is Better for RL:
- **H** = What color (independent action)
- **S** = How pure the color is (independent action)
- **V** = How bright (independent action)

Each channel can be adjusted independently!

## The sRGB Nonlinearity — Complete Transfer Function Derivation

sRGB is the most common color space in digital imaging. Its nonlinear transfer function is more complex than simple gamma and requires careful treatment.

### Step 1: The sRGB Standard (IEC 61966-2-1)

**Forward transfer (linear → sRGB):**
$$C_{\text{sRGB}} = \begin{cases} 12.92 \cdot C_{\text{linear}} & C_{\text{linear}} \leq 0.0031308 \\ 1.055 \cdot C_{\text{linear}}^{1/2.4} - 0.055 & C_{\text{linear}} > 0.0031308 \end{cases}$$

**Inverse transfer (sRGB → linear):**
$$C_{\text{linear}} = \begin{cases} C_{\text{sRGB}} / 12.92 & C_{\text{sRGB}} \leq 0.04045 \\ \left(\frac{C_{\text{sRGB}} + 0.055}{1.055}\right)^{2.4} & C_{\text{sRGB}} > 0.04045 \end{cases}$$

### Step 2: Derivation of the Piecewise Breakpoint

**Why 0.0031308?** The linear segment and the power curve must meet smoothly (continuous and with matching derivatives):

At the junction point $C_0$:

**Continuity:** $12.92 \cdot C_0 = 1.055 \cdot C_0^{1/2.4} - 0.055$

**Derivative matching:** $12.92 = \frac{1.055}{2.4} \cdot C_0^{1/2.4 - 1}$

From the derivative equation:
$$C_0 = \left(\frac{12.92 \times 2.4}{1.055}\right)^{2.4/(1/2.4 - 1)} = \left(\frac{31.008}{1.055}\right)^{-2.4/1.4167}$$

Solving numerically: $C_0 \approx 0.0031308$, giving $C_{\text{sRGB}} = 12.92 \times 0.0031308 \approx 0.04045$. $\blacksquare$

### Step 3: Why Not Pure $\gamma = 2.2$?

A pure power law $C^{1/\gamma}$ has an **infinite derivative** at $C = 0$:

$$\frac{d}{dC}C^{1/\gamma}\bigg|_{C=0} = \frac{1}{\gamma} C^{1/\gamma - 1}\bigg|_{C=0} = \infty \quad (\text{since } 1/\gamma < 1)$$

This means tiny changes in near-black values would map to large sRGB differences — wasting quantization levels. The linear segment replaces this problematic region:

$$\frac{d}{dC}(12.92 \cdot C)\bigg|_{C=0} = 12.92 \quad (\text{finite!})$$

### Step 4: Effective Gamma

The sRGB transfer function is often approximated as $\gamma = 2.2$, but the true effective gamma varies:
- Near black ($C < 0.003$): effective $\gamma = 1.0$ (linear)
- Mid-tones ($C \approx 0.2$): effective $\gamma \approx 2.4$
- Highlights ($C \approx 1.0$): effective $\gamma \approx 2.2$

The end-to-end gamma (from the full curve) averages to approximately $2.2$, which is why this value is commonly cited.

### Step 5: Why RL Agents Should Work in Linear Space

Image operations (blending, filtering, compositing) are physically correct only in linear space:

$$\text{Correct:} \quad I_{\text{blend}} = \alpha \cdot I_1^{\text{linear}} + (1-\alpha) \cdot I_2^{\text{linear}}$$

$$\text{Incorrect:} \quad I_{\text{blend}} = \alpha \cdot I_1^{\text{sRGB}} + (1-\alpha) \cdot I_2^{\text{sRGB}}$$

**Proof of incorrectness:** The sRGB transform is nonlinear ($C^{2.4}$), so $\alpha f(x) + (1-\alpha)f(y) \neq f(\alpha x + (1-\alpha)y)$. Operating in sRGB space computes the wrong physical mixture. $\blacksquare$

For RL agents: linearize inputs before processing, then re-encode to sRGB for display.

## Chromatic Adaptation — The Von Kries Model

Human vision adapts to the color of illumination (white balance). The Von Kries model explains how and provides the mathematical basis for computational white balancing.

### Step 1: The Problem of Illuminant Change

The same object under different light sources produces different RGB values:
$$\mathbf{c}_{\text{observed}} = \mathbf{E}_{\text{illuminant}} \odot \mathbf{r}_{\text{surface}}$$

where $\odot$ is element-wise multiplication in the cone response space, and $\mathbf{E}$ represents the illuminant's effect on each cone type.

### Step 2: Von Kries Adaptation Model

**Hypothesis (Von Kries, 1902):** The visual system adapts by independently scaling the response of each cone type:

$$\begin{bmatrix} L_{\text{adapted}} \\ M_{\text{adapted}} \\ S_{\text{adapted}} \end{bmatrix} = \begin{bmatrix} k_L & 0 & 0 \\ 0 & k_M & 0 \\ 0 & 0 & k_S \end{bmatrix} \begin{bmatrix} L \\ M \\ S \end{bmatrix}$$

The gain factors are determined by the illuminant's cone responses:
$$k_L = \frac{L_{\text{white}}^{\text{target}}}{L_{\text{white}}^{\text{source}}}, \quad k_M = \frac{M_{\text{white}}^{\text{target}}}{M_{\text{white}}^{\text{source}}}, \quad k_S = \frac{S_{\text{white}}^{\text{target}}}{S_{\text{white}}^{\text{source}}}$$

### Step 3: Implementation in Practice (Bradford Transform)

Since we work in XYZ (not LMS cone space), we need the full pipeline:

$$\mathbf{c}_{\text{adapted}} = \mathbf{M}_{\text{XYZ→LMS}}^{-1} \cdot \mathbf{D} \cdot \mathbf{M}_{\text{XYZ→LMS}} \cdot \mathbf{c}_{\text{source}}$$

The Bradford matrix $\mathbf{M}$ (empirically optimized):
$$\mathbf{M}_{\text{Bradford}} = \begin{bmatrix} 0.8951 & 0.2664 & -0.1614 \\ -0.7502 & 1.7135 & 0.0367 \\ 0.0389 & -0.0685 & 1.0296 \end{bmatrix}$$

### Step 4: Proof That Von Kries Is the Optimal Diagonal Model

**Theorem:** Among all diagonal adaptation matrices in cone space, Von Kries minimizes the color error under illuminant change.

**Proof:** For diagonal $\mathbf{D} = \text{diag}(d_1, d_2, d_3)$, the adaptation error for a surface $\mathbf{r}$ is:

$$\epsilon = \|\mathbf{D} \cdot \mathbf{E}_s \odot \mathbf{r} - \mathbf{E}_t \odot \mathbf{r}\|^2 = \sum_{i=1}^3 (d_i E_{s,i} - E_{t,i})^2 r_i^2$$

Minimizing over $d_i$: $\frac{\partial \epsilon}{\partial d_i} = 2(d_i E_{s,i} - E_{t,i}) E_{s,i} r_i^2 = 0$

$$d_i = \frac{E_{t,i}}{E_{s,i}} \quad \blacksquare$$

This recovers the Von Kries gain factors exactly.

### Step 5: Common Illuminant White Points

| Illuminant | CCT (K) | $x$ | $y$ | Description |
|:-----------|:--------|:----|:----|:------------|
| A | 2856 | 0.4476 | 0.4074 | Incandescent tungsten |
| D50 | 5003 | 0.3457 | 0.3585 | Horizon daylight |
| D65 | 6504 | 0.3127 | 0.3290 | Noon daylight (sRGB reference) |
| F2 | 4230 | 0.3721 | 0.3751 | Cool white fluorescent |

### RL Application: Automatic White Balance

An RL agent for automatic white balance estimates the illuminant and applies Von Kries correction. The action space is the gain vector $(k_L, k_M, k_S)$, and the reward measures color accuracy under the estimated illuminant.

## Color Difference Metrics — From ΔE*ab to CIEDE2000

Measuring color difference precisely is essential for RL reward design in color manipulation tasks.

### Step 1: The Original CIE76 Color Difference

$$\Delta E_{76}^* = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$$

This is simply the Euclidean distance in CIELAB space. JND $\approx 2.3$ units.

### Step 2: Limitations of ΔE*76

Despite CIELAB's design goal of perceptual uniformity, experiments revealed systematic non-uniformities:
- **Blue region:** JND ellipses are elongated along the hue direction → $\Delta E_{76}$ underestimates hue differences
- **Chroma dependency:** High-chroma colors have larger JND → $\Delta E_{76}$ overestimates differences at high saturation
- **Lightness dependency:** Dark colors are harder to distinguish → $\Delta E_{76}$ overestimates differences in dark regions

### Step 3: CMC(l:c) Color Difference (1984)

The CMC formula uses correction factors based on lightness and chroma:

$$\Delta E_{CMC} = \sqrt{\left(\frac{\Delta L^*}{l \cdot S_L}\right)^2 + \left(\frac{\Delta C^*}{c \cdot S_C}\right)^2 + \left(\frac{\Delta H^*}{S_H}\right)^2}$$

where $S_L, S_C, S_H$ are weighting functions that depend on the color's position in CIELAB:

$$S_L = \begin{cases} 0.511 & L^* < 16 \\ 0.040975 L^* / (1 + 0.01765 L^*) & L^* \geq 16 \end{cases}$$

### Step 4: CIEDE2000 — The Current Standard

CIEDE2000 adds five corrections to CIELAB distance:

$$\Delta E_{00} = \sqrt{\left(\frac{\Delta L'}{k_L S_L}\right)^2 + \left(\frac{\Delta C'}{k_C S_C}\right)^2 + \left(\frac{\Delta H'}{k_H S_H}\right)^2 + R_T \frac{\Delta C'}{k_C S_C} \frac{\Delta H'}{k_H S_H}}$$

The corrections address:
1. **Lightness weighting** ($S_L$): adjusts for dark/light asymmetry
2. **Chroma weighting** ($S_C$): adjusts for saturation-dependent sensitivity
3. **Hue weighting** ($S_H$): adjusts for hue-angle-dependent sensitivity
4. **Chroma modification** ($a' = a^*(1 + G)$): stretches the $a^*$ axis near the neutral axis
5. **Rotation term** ($R_T$): rotates the major axis of the JND ellipse in the blue region

### Step 5: Practical Impact on RL Rewards

**Comparison of metrics for a blue hue shift of $\Delta h = 5°$ at $C^* = 50, L^* = 50$:**

| Metric | Value | Perceptual Assessment |
|:-------|:------|:---------------------|
| $\Delta E_{76}$ | 4.4 | Over-penalizes (blue hue shifts are less visible) |
| $\Delta E_{CMC}$ | 2.8 | Better, but still imperfect |
| $\Delta E_{00}$ | 1.9 | Matches human perception well |

For an RL color correction agent, using $\Delta E_{00}$ as the reward function leads to behaviors aligned with human preferences, while $\Delta E_{76}$ would waste effort on perceptually unimportant corrections in the blue region.

In [ ]:
# Create a rich sample image
def create_colorful_image(size=128):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Create color wheel pattern
    for i in range(size):
        for j in range(size):
            x = j - size//2
            y = i - size//2
            angle = np.arctan2(y, x) * 180 / np.pi + 180  # 0-360
            radius = np.sqrt(x**2 + y**2) / (size//2)
            if radius <= 1.0:
                h = int(angle / 2)  # 0-180 for OpenCV
                s = int(radius * 255)
                v = 255
                hsv_pixel = np.array([[[h, s, v]]], dtype=np.uint8)
                rgb_pixel = cv2.cvtColor(hsv_pixel, cv2.COLOR_HSV2RGB)
                img[i, j] = rgb_pixel[0, 0]
    return img

color_img = create_colorful_image(128)

# Convert to different color spaces
hsv_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2HSV)
lab_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2LAB)

# Visualize all color spaces
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

# RGB
axes[0, 0].imshow(color_img)
axes[0, 0].set_title('Original RGB')
axes[0, 1].imshow(color_img[:,:,0], cmap='Reds')
axes[0, 1].set_title('R Channel')
axes[0, 2].imshow(color_img[:,:,1], cmap='Greens')
axes[0, 2].set_title('G Channel')
axes[0, 3].imshow(color_img[:,:,2], cmap='Blues')
axes[0, 3].set_title('B Channel')

# HSV
axes[1, 0].imshow(color_img)
axes[1, 0].set_title('Original (for reference)')
axes[1, 1].imshow(hsv_img[:,:,0], cmap='hsv')
axes[1, 1].set_title('H (Hue) - Color Type')
axes[1, 2].imshow(hsv_img[:,:,1], cmap='gray')
axes[1, 2].set_title('S (Saturation) - Purity')
axes[1, 3].imshow(hsv_img[:,:,2], cmap='gray')
axes[1, 3].set_title('V (Value) - Brightness')

# LAB
axes[2, 0].imshow(color_img)
axes[2, 0].set_title('Original (for reference)')
axes[2, 1].imshow(lab_img[:,:,0], cmap='gray')
axes[2, 1].set_title('L (Lightness)')
axes[2, 2].imshow(lab_img[:,:,1], cmap='RdYlGn_r')
axes[2, 2].set_title('a (Green↔Red)')
axes[2, 3].imshow(lab_img[:,:,2], cmap='PuOr')
axes[2, 3].set_title('b (Blue↔Yellow)')

for ax in axes.flat:
    ax.axis('off')

axes[0, 0].set_ylabel('RGB', fontsize=14, rotation=0, labelpad=50)
axes[1, 0].set_ylabel('HSV', fontsize=14, rotation=0, labelpad=50)
axes[2, 0].set_ylabel('LAB', fontsize=14, rotation=0, labelpad=50)

plt.suptitle('Color Spaces Comparison — Same Image, Different Representations', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('color_spaces_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## CIELAB ΔE — Proof That It Approximates Perceptual Difference

The CIELAB color space was designed so that Euclidean distance corresponds to perceived color difference. Here we derive why.

### Step 1: The Perceptual Uniformity Goal

**Requirement:** A distance $\Delta E_{ab}^* = \sqrt{(\Delta L^*)^2 + (\Delta a^*)^2 + (\Delta b^*)^2}$ of approximately 2.3 should correspond to a just-noticeable difference (JND) everywhere in color space.

### Step 2: Why the Cube Root in LAB?

The L* formula: $L^* = 116 f(Y/Y_n) - 16$ where $f(t) = t^{1/3}$ for $t > \delta^3$.

**Derivation from Stevens' Power Law:**

Human brightness perception follows: $\text{Perceived brightness} \propto L^{0.33}$

For a small change $\Delta Y$ to be equally perceptible everywhere:
$$\frac{dL^*}{dY} \cdot \Delta Y = \text{const} \quad \text{(constant perceptual step)}$$

If we want equal perceptual steps for equal physical ratios ($\Delta Y / Y = \text{const}$), then:
$$\frac{dL^*}{dY} \propto \frac{1}{Y}$$

$$L^* \propto \ln Y \quad \text{(Weber-Fechner)}$$

Stevens' refinement gives: $L^* \propto Y^{1/3}$, which better fits experimental data. With the constants $116$ and $-16$ chosen so that:
- $L^* = 0$ for black ($Y = 0$)
- $L^* = 100$ for white ($Y = Y_n$, the reference white)

### Step 3: The a* and b* Opponent Color Channels

$$a^* = 500\left[f(X/X_n) - f(Y/Y_n)\right] \quad \text{(red-green axis)}$$
$$b^* = 200\left[f(Y/Y_n) - f(Z/Z_n)\right] \quad \text{(yellow-blue axis)}$$

**Derivation from opponent color theory:** The visual system processes color as three opponent channels:
1. Luminance: $L \approx Y$ (achromatic)
2. Red-Green: $a \propto L_{\text{cones}} - M_{\text{cones}} \propto X - Y$
3. Blue-Yellow: $b \propto S_{\text{cones}} - (L + M) \propto Y - Z$

The cube-root nonlinearity is applied before differencing to account for the compressive response of each cone type individually.

### Step 4: Proof That ΔE Is Approximately Perceptually Uniform

**MacAdam Ellipses (1942):** MacAdam measured the JND ellipses in CIE xy chromaticity — regions within which color differences are imperceptible. In CIE xy, these ellipses vary wildly in size and orientation.

**Key result:** In CIELAB space, MacAdam ellipses become approximately circular with approximately constant radius:

$$\Delta E_{JND} \approx 2.3 \pm 0.5$$

across the gamut. The residual non-uniformity (±20%) led to improved formulas (CIEDE2000):

$$\Delta E_{00} = \sqrt{\left(\frac{\Delta L'}{k_L S_L}\right)^2 + \left(\frac{\Delta C'}{k_C S_C}\right)^2 + \left(\frac{\Delta H'}{k_H S_H}\right)^2 + R_T\frac{\Delta C'}{k_C S_C}\frac{\Delta H'}{k_H S_H}}$$

where $S_L, S_C, S_H$ are weighting functions and $R_T$ is a rotation term for the blue region.

### Step 5: RL Reward Design Implication

For an RL agent optimizing image color quality, the reward should use $\Delta E$ in CIELAB (not Euclidean distance in RGB):

$$R = -\Delta E_{ab}^*(I_{\text{output}}, I_{\text{target}}) = -\frac{1}{N}\sum_{i} \sqrt{(\Delta L_i^*)^2 + (\Delta a_i^*)^2 + (\Delta b_i^*)^2}$$

This ensures the agent optimizes for what humans actually perceive, not what is mathematically convenient.

## Gamut Mapping — Convex Hull Theory of Color Reproduction

Not all perceivable colors can be displayed on a monitor or printed. Gamut mapping addresses this fundamental limitation using convex geometry.

### Step 1: Color Gamut as a Convex Set

**Definition:** The gamut of a display is the set of all colors it can produce:

$$\mathcal{G} = \left\{\mathbf{c} = r\mathbf{R} + g\mathbf{G} + b\mathbf{B} : r, g, b \in [0, 1]\right\}$$

where $\mathbf{R}, \mathbf{G}, \mathbf{B}$ are the chromaticities of the display's primaries in CIE XYZ.

**Theorem:** The gamut is a convex set (in fact, a convex polytope).

**Proof:** For any two colors $\mathbf{c}_1, \mathbf{c}_2 \in \mathcal{G}$ and $\alpha \in [0, 1]$:
$$\alpha\mathbf{c}_1 + (1-\alpha)\mathbf{c}_2 = \alpha(r_1\mathbf{R} + g_1\mathbf{G} + b_1\mathbf{B}) + (1-\alpha)(r_2\mathbf{R} + g_2\mathbf{G} + b_2\mathbf{B})$$
$$= [\alpha r_1 + (1-\alpha)r_2]\mathbf{R} + [\alpha g_1 + (1-\alpha)g_2]\mathbf{G} + [\alpha b_1 + (1-\alpha)b_2]\mathbf{B}$$

Since $\alpha r_1 + (1-\alpha)r_2 \in [0,1]$ (convex combination of values in $[0,1]$), this point is in $\mathcal{G}$. $\blacksquare$

### Step 2: Out-of-Gamut Detection

A color $\mathbf{c}$ is out-of-gamut if it lies outside $\mathcal{G}$. In sRGB, this means:

$$\mathbf{c} \notin \mathcal{G}_{sRGB} \iff R < 0 \text{ or } R > 1 \text{ or } G < 0 \text{ or } G > 1 \text{ or } B < 0 \text{ or } B > 1$$

(after converting to the sRGB linear color space)

### Step 3: Gamut Mapping Strategies

**1. Clipping (simplest):** Project to the nearest point on the gamut boundary:
$$\mathbf{c}_{\text{clipped}} = \arg\min_{\mathbf{c}' \in \mathcal{G}} \|\mathbf{c} - \mathbf{c}'\|$$

In sRGB: simply clip each channel to $[0, 1]$. Fast but can cause hue shifts.

**2. Perceptual compression:** Compress the entire color range to fit within the gamut while preserving relative differences:
$$\mathbf{c}_{\text{mapped}} = f(\mathbf{c}) \quad \text{where } f: \mathbb{R}^3 \to \mathcal{G} \text{ is smooth and monotone}$$

**3. Chroma reduction (preserve lightness and hue):** In CIELAB polar coordinates $(L^*, C^*, h^*)$:
$$C^*_{\text{mapped}} = \min(C^*, C^*_{\text{boundary}}(L^*, h^*))$$

This preserves lightness and hue while reducing saturation — generally the most perceptually pleasing approach.

### Step 4: Gamut Volume as a Quality Metric

The gamut volume in CIELAB measures a display's color reproduction capability:
$$V_{\text{gamut}} = \iiint_{\mathcal{G}} dL^* \, da^* \, db^*$$

**Typical volumes:**
| Display | Gamut Volume (×10³ CIELAB units) | Coverage of visible gamut |
|:--------|:----------------------------------|:-------------------------|
| sRGB | ~820 | ~35% |
| Adobe RGB | ~1,210 | ~50% |
| DCI-P3 | ~1,100 | ~45% |
| Rec. 2020 | ~2,380 | ~76% |

### Step 5: RL for Gamut Mapping

An RL agent for cross-device color management learns a gamut mapping function:
- **State:** $(L^*, C^*, h^*, \text{source gamut, target gamut})$
- **Action:** adjusted $(L'^*, C'^*, h'^*)$
- **Reward:** perceptual quality $-\Delta E_{00}$ on test images

The agent outperforms fixed mappings by adapting to image content — e.g., preserving saturated skies in landscape photos while allowing chroma reduction in less important regions.

## Channel Decorrelation and the YCbCr Color Space

### Step 1: Why Decorrelation Matters

In natural images, the R, G, B channels are highly correlated ($\rho_{RG} \approx 0.9$). This means:
- Much of the information in G is already contained in R
- Storing all three channels separately is wasteful
- For RL, correlated state dimensions slow convergence

### Step 2: RGB to YCbCr Transform

The ITU-R BT.601 standard defines:

$$\begin{bmatrix} Y \\ C_b \\ C_r \end{bmatrix} = \begin{bmatrix} 0.299 & 0.587 & 0.114 \\ -0.169 & -0.331 & 0.500 \\ 0.500 & -0.419 & -0.081 \end{bmatrix} \begin{bmatrix} R \\ G \\ B \end{bmatrix} + \begin{bmatrix} 0 \\ 128 \\ 128 \end{bmatrix}$$

**Y** (luminance): weighted sum matching human brightness perception
**Cb** (blue-difference chrominance): $C_b \propto B - Y$
**Cr** (red-difference chrominance): $C_r \propto R - Y$

### Step 3: Proof That Chrominance Has Lower Bandwidth

**Theorem:** Human spatial resolution for chrominance is approximately 4× lower than for luminance.

**Evidence from psychophysics:** The contrast sensitivity function (CSF) for color has a cutoff frequency of ~10 cycles/degree, while for luminance it extends to ~40 cycles/degree.

**Mathematical consequence:** Chrominance can be subsampled by 2× in each dimension without perceptible loss:

$$\text{4:2:0 subsampling:} \quad Y: H \times W, \quad C_b: H/2 \times W/2, \quad C_r: H/2 \times W/2$$

**Data reduction:** $HW + HW/4 + HW/4 = 1.5HW$ (50% of full RGB = $3HW$)

### Step 4: Comparison of Decorrelation Transforms

| Transform | Type | Decorrelation | Complexity |
|:----------|:-----|:-------------|:-----------|
| KLT (PCA) | Signal-dependent | Optimal | $O(N)$ + eigendecomp |
| YCbCr | Fixed | Near-optimal for natural images | $O(N)$ |
| YIQ | Fixed | NTSC standard | $O(N)$ |
| DCT | Fixed | Asymptotically optimal | $O(N\log N)$ |

**Theorem:** The KLT achieves the minimum coding gain (best decorrelation) among all orthogonal transforms. The DCT approaches KLT performance as the signal model approaches a first-order Markov process (which natural images approximate).

### Step 5: RL State Space Reduction via YCbCr

For a 64×64 RGB image:
- RGB state: $64 \times 64 \times 3 = 12{,}288$ dimensions
- YCbCr with 4:2:0: $64 \times 64 + 32 \times 32 + 32 \times 32 = 6{,}144$ dimensions
- **50% state space reduction** with negligible perceptual loss

For deep RL, this means smaller networks, faster training, and better sample efficiency — a free lunch courtesy of color science.

### Step 6: Why JPEG Uses YCbCr + DCT

JPEG compression pipeline:
1. RGB → YCbCr (decorrelate channels)
2. Subsample Cb, Cr (exploit chrominance bandwidth limit)
3. 8×8 block DCT (decorrelate spatial correlations)
4. Quantize DCT coefficients (lossy — allocate more bits to low frequencies)
5. Entropy code (lossless compression of quantized coefficients)

Each step exploits a mathematical property: color correlation, bandwidth asymmetry, spatial correlation, frequency masking, and statistical redundancy.

## 3. Color Space Selection for RL Tasks

| Task | Best Color Space | Reason |
|:-----|:----------------|:-------|
| Brightness adjustment | HSV or LAB | V/L channel is independent |
| Color correction | LAB | Perceptually uniform |
| Object detection | RGB or HSV | Depends on object properties |
| Skin detection | YCbCr | Skin clusters well in CbCr |
| Night enhancement | HSV | Can boost V without color shift |

### For RL State Space:
$$s_t = \text{ColorSpace}(I_t) \in \mathbb{R}^{H \times W \times C}$$

Choosing the right color space **simplifies the RL problem** by making channels independent!

---
**Next:** Module 1.4 — Image as Matrix (Linear Algebra perspective)